In [ ]:
import textwrap

class Cliente:
    def __init__(self, nome, cpf, data_nascimento, endereco):
        self.nome = nome
        self.cpf = cpf
        self.data_nascimento = data_nascimento
        self.endereco = endereco
        self.contas = []

    def adicionar_conta(self, conta):
        self.contas.append(conta)

class PessoaFisica(Cliente):
    pass

class Conta:
    def __init__(self, agencia, numero, cliente):
        self.saldo = 0.0
        self.numero = numero
        self.agencia = agencia
        self.cliente = cliente
        self.historico = Historico()

    def depositar(self, valor):
        if valor > 0:
            self.saldo += valor
            self.historico.adicionar_transacao(Deposito(valor))
            return True
        return False

    def sacar(self, valor):
        if valor > 0 and valor <= self.saldo:
            self.saldo -= valor
            self.historico.adicionar_transacao(Saque(valor))
            return True
        return False

class ContaCorrente(Conta):
    def __init__(self, agencia, numero, cliente, limite, limite_saques):
        super().__init__(agencia, numero, cliente)
        self.limite = limite
        self.limite_saques = limite_saques
        self.numero_saques = 0

    def sacar(self, valor):
        if self.numero_saques >= self.limite_saques:
            print("\n@@@ Operação falhou! Limite de saques diários excedido. @@@")
            return False
        if valor > 0 and valor <= self.saldo:
            self.saldo -= valor
            self.historico.adicionar_transacao(Saque(valor))
            self.numero_saques += 1
            return True
        print("\n@@@ Operação falhou! Saldo insuficiente ou valor inválido. @@@")
        return False

class Historico:
    def __init__(self):
        self.transacoes = []

    def adicionar_transacao(self, transacao):
        self.transacoes.append(transacao)

class Transacao:
    def __init__(self, valor):
        self.valor = valor

class Deposito(Transacao):
    pass

class Saque(Transacao):
    pass

def formatar_saldo(valor):
    return f"R$ {valor:,.2f}".replace('.', ',').replace(',', '.')

def cadastrar_usuario(usuarios):
    cpf = input("Informe o CPF (somente números): ")
    usuario = filtrar_usuario(cpf, usuarios)

    if usuario:
        print("\n@@@ Já existe usuário com esse CPF! @@@")
        return

    nome = input("Informe o nome completo: ")
    data_nascimento = input("Informe a data de nascimento (dd-mm-aaaa): ")
    endereco = input("Informe o endereço (logradouro, nro - bairro - cidade/sigla estado): ")

    novo_usuario = PessoaFisica(nome, cpf, data_nascimento, endereco)
    usuarios.append(novo_usuario)

    print("=== Usuário criado com sucesso! ===")

def criar_conta(agencia, numero_conta, usuarios):
    cpf = input("Informe o CPF do usuário: ")
    usuario = filtrar_usuario(cpf, usuarios)

    if usuario:
        nova_conta = ContaCorrente(agencia, numero_conta, usuario, limite=500, limite_saques=3)
        usuario.adicionar_conta(nova_conta)
        print("\n=== Conta criada com sucesso! ===")
        return nova_conta

    print("\n@@@ Usuário não encontrado, fluxo de criação de conta encerrado! @@@")

def filtrar_usuario(cpf, usuarios):
    for usuario in usuarios:
        if usuario.cpf == cpf:
            return usuario
    return None

def listar_contas(contas):
    for conta in contas:
        linha = f"""\
            Agência:\t{conta.agencia}
            C/C:\t\t{conta.numero}
            Titular:\t{conta.cliente.nome}
        """
        print("=" * 100)
        print(textwrap.dedent(linha))

def main():
    AGENCIA = "0001"
    usuarios = []
    contas = []

    while True:
        opcao = exibir_menu()

        if opcao == "d":
            cpf = input("Informe o CPF do titular da conta: ")
            usuario = filtrar_usuario(cpf, usuarios)
            if usuario:
                valor = float(input("Informe o valor do depósito: "))
                for conta in usuario.contas:
                    if conta.depositar(valor):
                        print("\n=== Depósito realizado com sucesso! ===")
                    else:
                        print("\n@@@ Operação falhou! O valor informado é inválido. @@@")

        elif opcao == "s":
            cpf = input("Informe o CPF do titular da conta: ")
            usuario = filtrar_usuario(cpf, usuarios)
            if usuario:
                valor = float(input("Informe o valor do saque: "))
                for conta in usuario.contas:
                    if conta.sacar(valor):
                        print("\n=== Saque realizado com sucesso! ===")

        elif opcao == "e":
            cpf = input("Informe o CPF do titular da conta: ")
            usuario = filtrar_usuario(cpf, usuarios)
            if usuario:
                for conta in usuario.contas:
                    exibir_extrato(conta)

        elif opcao == "nu":
            cadastrar_usuario(usuarios)

        elif opcao == "nc":
            numero_conta = len(contas) + 1
            conta = criar_conta(AGENCIA, numero_conta, usuarios)

            if conta:
                contas.append(conta)

        elif opcao == "lc":
            listar_contas(contas)

        elif opcao == "q":
            break

        else:
            print("Operação inválida, por favor selecione novamente a operação desejada.")

def exibir_extrato(conta):
    print("\n================ EXTRATO ================")
    if not conta.historico.transacoes:
        print("Não foram realizadas movimentações.")
    else:
        for transacao in conta.historico.transacoes:
            tipo = "Depósito" if isinstance(transacao, Deposito) else "Saque"
            print(f"{tipo}:\t{formatar_saldo(transacao.valor)}")
    print(f"\nSaldo:\t\t{formatar_saldo(conta.saldo)}")
    print("==========================================")

def exibir_menu():
    menu = """\n
    ================ MENU ================
    [d]\tDepositar
    [s]\tSacar
    [e]\tExtrato
    [nu]\tNovo Usuário
    [nc]\tNova Conta
    [lc]\tListar Contas
    [q]\tSair
    => """
    return input(textwrap.dedent(menu))

if __name__ == "__main__":
    main()



================ MENU ================
[d]	Depositar
[s]	Sacar
[e]	Extrato
[nu]	Novo Usuário
[nc]	Nova Conta
[lc]	Listar Contas
[q]	Sair
=> q
